In [51]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Literal
from dotenv import load_dotenv
from langchain_huggingface import ChatHuggingFace,HuggingFaceEndpoint
import os
from pydantic import BaseModel,Field

In [52]:
load_dotenv()


True

In [53]:
llm=HuggingFaceEndpoint(
     repo_id="Qwen/Qwen3-8B",
        task="text-generation",
        max_new_tokens=512,
        temperature=0,
     
)
model=ChatHuggingFace(llm=llm)

In [ ]:
answer=model.invoke("number of alphabets in English Language")
print(answer.content)



The English language uses **26 alphabets** (letters). These are the letters from **A to Z** (both uppercase and lowercase). 

### Key Points:
- **Total Letters**: 26 (A, B, C, D, E, F, G, H, I, J, K, L, M, N, O, P, Q, R, S, T, U, V, W, X, Y, Z).
- **Case Sensitivity**: While uppercase (A–Z) and lowercase (a–z) are distinct in writing, they are considered part of the same alphabet.
- **No Diacritics**: The English alphabet does not include letters with accents (e.g., é, ñ) or special characters like numbers or symbols.

This standard set of letters is used for spelling, writing, and communication in English.


In [55]:
class SentimentSchema(BaseModel):
    sentiment:Literal['positive','negative']=Field(description="sentiment of the review")

In [56]:
class DiagonosisModel(BaseModel):
    issue_type:Literal['UX','Performance','Bug','Other']=Field(description="The category of the issue mentioned in the review")
    tone: Literal["angry", "frustrated", "disappointed", "calm"] = Field(description='The emotional tone expressed by the user')
    urgency: Literal["low", "medium", "high"] = Field(description='How urgent or critical the issue appears to be')

In [57]:
structured_model = model.with_structured_output(
    SentimentSchema,
    method="json_schema",
#include_raw=True
)

structured_model2 = model.with_structured_output(
    DiagonosisModel,
    method="json_schema"
)

In [59]:
prompt = """
Classify the sentiment of the following review.

You MUST return exactly one of these values:
positive
negative

Review:
The software is too good
"""

result = structured_model.invoke(prompt)

print(result)

{'positive': 'The software is too good'}


In [60]:
class ReviewState(TypedDict):

    review: str
    sentiment: Literal["positive", "negative"]
    diagnosis: dict
    response: str


In [61]:
def find_sentiment(state: ReviewState):

    prompt = f'For the following review find out the sentiment \n {state["review"]}'
    sentiment = structured_model.invoke(prompt).sentiment

    return {'sentiment': sentiment}

def check_sentiment(state: ReviewState) -> Literal["positive_response", "run_diagnosis"]:

    if state['sentiment'] == 'positive':
        return 'positive_response'
    else:
        return 'run_diagnosis'
    
def positive_response(state: ReviewState):

    prompt = f"""Write a warm thank-you message in response to this review:
    \n\n\"{state['review']}\"\n
Also, kindly ask the user to leave feedback on our website."""
    
    response = model.invoke(prompt).content

    return {'response': response}

def run_diagnosis(state: ReviewState):

    prompt = f"""Diagnose this negative review:\n\n{state['review']}\n"
    "Return issue_type, tone, and urgency.
"""
    response = structured_model2.invoke(prompt)

    return {'diagnosis': response.model_dump()}

def negative_response(state: ReviewState):

    diagnosis = state['diagnosis']

    prompt = f"""You are a support assistant.
The user had a '{diagnosis['issue_type']}' issue, sounded '{diagnosis['tone']}', and marked urgency as '{diagnosis['urgency']}'.
Write an empathetic, helpful resolution message.
"""
    response = model.invoke(prompt).content

    return {'response': response}

In [ ]:
graph=StateGraph(ReviewState)

graph.add_node('find_sentiment',find_sentiment)
graph.add_node('positive_response',positive_response)
graph.add_node('negative_response',negative_response)
graph.add_node('run_diagnosis',run_diagnosis)
graph.add_edge(START,)